# Audit qualité des CSV – accidents de la route 2019-2025

Objectif : vérifier les 28 fichiers CSV de l'ONISR **avant** de les nettoyer dans MySQL.
On repère les problèmes ici en Python, puis on les corrige en SQL dans Navicat.

Plan :
1. on lit un fichier pour voir les données
2. on fait les premiers contrôles à la main
3. on contrôle les coordonnées GPS
4. on contrôle un fichier lieux
5. on range tous les contrôles dans une fonction
6. on lance la fonction sur les 28 fichiers et on enregistre le rapport

## 0. Préparation

In [ ]:
# on charge les bibliothèques dont on a besoin
import re                    # re : pour retrouver l'année dans le nom des fichiers
from pathlib import Path     # Path : pour parcourir les fichiers d'un dossier
import pandas as pd          # pandas : pour lire et travailler sur des tableaux (les csv)

# on indique où sont rangés les csv
dossier = Path("C:/Users/elise/OneDrive/Bureau/formation/16_Dataviz/données")

# on vérifie qu'on trouve bien les fichiers
sorted(f.name for f in dossier.glob("*.csv"))

## 1. On lit un fichier pour voir à quoi ressemblent les données

In [ ]:
# on lit un seul fichier pour commencer
# sep=";" : les colonnes sont séparées par des points-virgules
# dtype=str : on garde tout en texte, on ne veut rien transformer pendant l'audit
# keep_default_na=False : les cases vides restent du texte vide "" pour pouvoir les compter
df = pd.read_csv(dossier / "caract-2025.csv", sep=";", dtype=str, keep_default_na=False)

# on affiche les 5 premières lignes
df.head()

## 2. Premiers contrôles à la main

In [ ]:
# taille du tableau : (nombre de lignes, nombre de colonnes)
print("lignes, colonnes :", df.shape)
print("noms des colonnes :", list(df.columns))

# duplicated() renvoie vrai quand le numéro est déjà apparu plus haut
# sum() compte les vrai (vrai = 1 et faux = 0)
print("doublons sur Num_Acc :", df["Num_Acc"].duplicated().sum())

# on compte les cases vides : le 1er sum() compte par colonne, le 2e fait le total
print("cases vides :", (df == "").sum().sum())

# certains -1 sont écrits " -1" avec un espace devant
# donc on enlève d'abord les espaces (strip) puis on compte les -1 (= non renseigné)
valeurs = df.apply(lambda colonne: colonne.str.strip())
print("valeurs -1 :", (valeurs == "-1").sum().sum())

## 3. On contrôle les coordonnées GPS

In [ ]:
# les coordonnées sont du texte avec une virgule ("48,86638600")
# on remplace la virgule par un point puis on convertit en nombre
# errors="coerce" : si la conversion est impossible, on met NaN (= vide) au lieu de planter
lat = pd.to_numeric(df["lat"].str.strip().str.replace(",", "."), errors="coerce")
lon = pd.to_numeric(df["long"].str.strip().str.replace(",", "."), errors="coerce")

# on ne vérifie que la métropole : les départements d'outre-mer commencent par 97 ou 98
# le ~ veut dire "pas" : on garde ce qui ne commence PAS par 97 ou 98
metropole = ~df["dep"].str.strip().str.match(r"^9[78]")

# un point est bon s'il est dans le rectangle de la France métropolitaine (Corse comprise)
dans_france = lat.between(41, 51.2) & lon.between(-5.5, 9.7)
hors_france = metropole & ~dans_france
print("coordonnées hors métropole :", hors_france.sum())

# si en échangeant latitude et longitude le point tombe en France, c'est une inversion
inversees = hors_france & lon.between(41, 51.2) & lat.between(-5.5, 9.7)
print("dont latitude et longitude inversées :", inversees.sum())

# on affiche les lignes concernées
df[hors_france]

## 4. On contrôle un fichier lieux

In [ ]:
lieux = pd.read_csv(dossier / "lieux-2025.csv", sep=";", dtype=str, keep_default_na=False)

# value_counts() compte combien de fois chaque Num_Acc apparaît
# normalement 1 fois : si c'est plus, l'accident a plusieurs lignes de lieux
nb_par_accident = lieux["Num_Acc"].value_counts()
print("accidents avec plusieurs lignes de lieux :", (nb_par_accident > 1).sum())

# vitesse max autorisée : on convertit en nombre
# puis on cherche les valeurs impossibles (moins de 5 ou plus de 130 km/h) sans compter les -1
vma = pd.to_numeric(lieux["vma"].str.strip(), errors="coerce")
incoherentes = ((vma < 5) | (vma > 130)) & (vma != -1)
print("vitesses incohérentes :", incoherentes.sum())
print(vma[incoherentes].value_counts())

## 5. On range tous les contrôles dans une fonction

Une **fonction** = un bloc de code qu'on écrit une fois et qu'on peut relancer sur n'importe quel fichier.
Ici, `auditer(chemin)` lit un fichier, fait les contrôles et renvoie les résultats sous forme de dictionnaire (des paires nom : valeur).

In [ ]:
def type_de_fichier(nom):
    """renvoie caract, lieux, vehicules ou usagers selon le nom du fichier"""
    nom = nom.lower()
    if "caract" in nom or "carcteristiques" in nom:   # attention : "carcteristiques" est mal écrit en 2021 et 2022
        return "caract"
    for t in ("lieux", "vehicules", "usagers"):
        if t in nom:
            return t
    return None   # ce n'est pas un des 4 types : on ne l'audite pas


def auditer(chemin):
    """lit un csv et renvoie un dictionnaire avec le résultat de chaque contrôle"""
    t = type_de_fichier(chemin.name)
    df = pd.read_csv(chemin, sep=";", dtype=str, keep_default_na=False)

    # en 2025 l'en-tête "id_vehicule" contient un retour à la ligne : on le repère puis on nettoie les noms
    entete_cassee = any("\n" in c for c in df.columns)
    df.columns = [c.strip() for c in df.columns]

    # en 2022 la clé s'appelle Accident_Id au lieu de Num_Acc : on prend la 1re colonne dans tous les cas
    cle = df.columns[0]

    # on retrouve l'année dans le nom du fichier (4 chiffres qui commencent par 20)
    annee = int(re.search(r"20\d\d", chemin.name).group())

    valeurs = df.apply(lambda colonne: colonne.str.strip())
    r = {
        "fichier": chemin.name,
        "type": t,
        "annee": annee,
        "nb_lignes": len(df),
        "nb_colonnes": df.shape[1],
        "nom_cle": cle,
        "entete_cassee": entete_cassee,
        "cles_mal_formees": (~df[cle].str.match(r"^\d{12}$")).sum(),   # un Num_Acc = 12 chiffres
        "cases_vides": (df == "").sum().sum(),
        "valeurs_moins_1": (valeurs == "-1").sum().sum(),
    }

    # contrôles propres aux fichiers caract
    if t == "caract":
        lat = pd.to_numeric(df["lat"].str.strip().str.replace(",", "."), errors="coerce")
        lon = pd.to_numeric(df["long"].str.strip().str.replace(",", "."), errors="coerce")
        metropole = ~df["dep"].str.strip().str.match(r"^9[78]")
        hors = metropole & ~(lat.between(41, 51.2) & lon.between(-5.5, 9.7))
        r["doublons_cle"] = df[cle].duplicated().sum()
        r["coord_hors_metropole"] = hors.sum()
        r["coord_inversees"] = (hors & lon.between(41, 51.2) & lat.between(-5.5, 9.7)).sum()
        r["dep_a_1_chiffre"] = df["dep"].str.strip().str.match(r"^\d$").sum()   # "1" au lieu de "01"

    # contrôles propres aux fichiers lieux
    if t == "lieux":
        vma = pd.to_numeric(df["vma"].str.strip(), errors="coerce")
        r["accidents_multi_lieux"] = (df[cle].value_counts() > 1).sum()
        r["vma_incoherente"] = (((vma < 5) | (vma > 130)) & (vma != -1)).sum()
        r["nbv_non_numerique"] = (~df["nbv"].str.strip().str.match(r"^-?\d+$")).sum()   # ex. "#VALEURMULTI"

    # contrôles propres aux fichiers vehicules et usagers
    if t in ("vehicules", "usagers"):
        # \u00a0 = l'espace insécable (invisible) qu'on trouve dans "155 975 242"
        r["id_vehicule_espaces_insecables"] = df["id_vehicule"].str.contains("\u00a0").sum()
        r["id_usager_present"] = "id_usager" in df.columns if t == "usagers" else None

    if t == "usagers":
        an = pd.to_numeric(df["an_nais"].str.strip(), errors="coerce")
        r["an_nais_avant_1910"] = ((an > 0) & (an < 1910)).sum()

    return r


# on teste la fonction sur un seul fichier
auditer(dossier / "caract-2025.csv")

## 6. On lance l'audit sur les 28 fichiers et on enregistre le rapport

In [ ]:
# on garde seulement les csv qui sont un des 4 types 
fichiers = sorted(f for f in dossier.glob("*.csv") if type_de_fichier(f.name))
print(len(fichiers), "fichiers à auditer")

# une boucle : on passe sur chaque fichier et on garde son résultat dans une liste
resultats = []
for f in fichiers:
    print("audit de", f.name)
    resultats.append(auditer(f))

# on transforme la liste de résultats en tableau : une ligne par fichier, une colonne par contrôle
rapport = pd.DataFrame(resultats).sort_values(["type", "annee"])

# on enregistre le rapport à côté des csv (utf-8-sig : pour qu'Excel affiche bien les accents)
rapport.to_csv(dossier / "audit_qualite.csv", sep=";", index=False, encoding="utf-8-sig")
rapport

In [ ]:
# vérification : nombre de lignes par année et par type de fichier
# ce sont les chiffres qu'on doit retrouver dans MySQL après l'import
rapport.pivot_table(index="annee", columns="type", values="nb_lignes", aggfunc="sum")

## Ce que l'audit a trouvé (et ce qu'on fait ensuite en SQL)

| Problème détecté | Correction dans MySQL (Navicat) |
|---|---|
| clé `Accident_Id` en 2022 | associée à `Num_Acc` à l'import |
| départements à 1 chiffre en 2019 | `LPAD(dep, 2, '0')` |
| coordonnées inversées ou impossibles | `CASE` sur latitude / longitude |
| plusieurs lignes de lieux par accident (2023-2025) | on garde la 1re ligne (`MIN(ligne_id)`) |
| vitesses incohérentes | mises à NULL |
| espaces insécables dans les identifiants | `REGEXP_REPLACE(…, '[^0-9]', '')` |
| `-1` et cases vides | mis à NULL |
| années de naissance avant 1910 | mises à NULL |